[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-advanced-clustering.ipynb)

# Advanced Clustering, Density & Graph Methods

*AIBits Academy · Machine Learning End To End · ⚠ Advanced Topic*

Three theoretically deeper techniques — introduced here at a foundational level. They sit past the core classical-ML curriculum, closer to Bayesian nonparametrics and graph theory.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Packages that Colab does not ship by default (a no-op if already installed)
%pip install -q scikit-fuzzy

> **⚠ Why This Page Is Marked "Advanced"**
>
> Everything below assumes you're comfortable with K-Means, GMM/EM, and basic probability. These three techniques are genuinely useful but sit at a higher mathematical bar than the rest of this course — treat this page as a foundational preview rather than a complete treatment.

## ⚠ Advanced: Dirichlet Process K-Means

Every clustering algorithm so far — K-Means, GMM, Hierarchical, DBSCAN — either requires you to specify k upfront or discovers clusters via density/linkage heuristics. The **Dirichlet Process (DP)** takes a different, Bayesian nonparametric approach: it places a prior over an *infinite* number of possible clusters, letting the number actually used grow naturally with the data, and to be inferred rather than chosen.

$$G \sim \mathrm{DP}(\alpha, G_0) \qquad \text{— a random distribution } G \text{ drawn from a Dirichlet Process with concentration } \alpha \text{ and base distribution } G_0$$

The concentration parameter α controls how eagerly new clusters are created: small α favours reusing existing clusters (few, larger clusters); large α favours spawning new ones (many, smaller clusters). In practice, this is implemented via the **Chinese Restaurant Process** — an intuitive metaphor where each new data point either joins an existing "table" (cluster) with probability proportional to that table's current size, or starts a new table with probability proportional to α.

In [ ]:
from sklearn.mixture import BayesianGaussianMixture
import numpy as np

# Bengaluru customer segments — true number of segments unknown in advance
np.random.seed(12)
c1 = np.random.normal([20,5], 2, (100,2))
c2 = np.random.normal([60,15], 3, (80,2))
c3 = np.random.normal([100,8], 2.5, (60,2))
X = np.vstack([c1,c2,c3])

# weight_concentration_prior (α) — small value favours FEWER effective clusters
dpgmm = BayesianGaussianMixture(
    n_components=10,               # upper bound — NOT the answer, just a ceiling
    weight_concentration_prior_type='dirichlet_process',
    weight_concentration_prior=0.5,   # small α → prefers reusing clusters
    random_state=42).fit(X)

# Effective clusters actually used: components with non-negligible weight
active = (dpgmm.weights_ > 0.05).sum()
print(f"Requested up to 10 components; DP inferred {active} were actually needed")
print(f"Cluster weights: {np.round(dpgmm.weights_, 3)}")

Note the pattern: you supply a generous upper bound (10), and the Dirichlet Process prior automatically shrinks unnecessary components' weights toward zero, leaving only the genuinely-supported clusters active — a fundamentally different answer to "how many clusters?" than the elbow method or BIC search from the GMM chapter, which require you to explicitly fit and compare many separate values of k.

## ⚠ Advanced: Kernel Density Estimation

Kernel Density Estimation (KDE) is a non-parametric way to estimate a probability density directly from data, without assuming any specific distributional family (unlike GMM, which assumes the data is a mixture of Gaussians specifically):

$$\hat{p}(x) = \dfrac{1}{nh}\sum_i K\!\left(\dfrac{x-x_i}{h}\right) \qquad K = \text{kernel function (usually Gaussian)},\ h = \text{bandwidth}$$

Every data point contributes a small "bump" (the kernel) centred on itself; the estimated density is the sum of all these bumps. The **bandwidth h** is the critical hyperparameter — too small and the estimate is spiky and overfit to individual points; too large and real structure gets smoothed away entirely.

In [ ]:
from sklearn.neighbors import KernelDensity
from sklearn.model_selection import GridSearchCV

# Surat textile order-value distribution — genuinely bimodal (retail + bulk orders)
orders = np.concatenate([np.random.normal(400,80,300), np.random.normal(2200,300,60)]).reshape(-1,1)

# Bandwidth selection via cross-validated grid search — analogous to hyperparameter tuning elsewhere
grid = GridSearchCV(KernelDensity(kernel='gaussian'),
                     {'bandwidth': np.logspace(1, 2.5, 20)}, cv=5)
grid.fit(orders)
kde = grid.best_estimator_
print(f"Best bandwidth: {kde.bandwidth:.1f}")

# Score new points: log-density (higher = more "typical" / expected)
test_points = np.array([[400], [1200], [2200]])
log_density = kde.score_samples(test_points)
for pt, ld in zip(test_points.ravel(), log_density):
    print(f"  order_value=₹{pt}   log-density={ld:.3f}   (₹1200 sits in the low-density VALLEY between modes)")

The Anomaly Detection chapter's Isolation Forest and One-Class SVM implicitly learn "normal" through partitioning or a boundary; KDE instead gives you the actual estimated probability density everywhere — a point in a genuine valley between two legitimate modes (like ₹1200 above, between typical retail and bulk order sizes) scores as unusual even though it isn't an "outlier" in the traditional sense, just a rare combination.

## ⚠ Advanced: PageRank Algorithm

PageRank — the algorithm behind early Google Search — solves a graph problem: given a network of nodes connected by directed edges, which nodes are most "important"? Its core insight is recursive: **a node is important if important nodes link to it.**

$$\mathrm{PR}(p) = \dfrac{1-d}{N} + d\cdot\sum_{q\to p} \dfrac{\mathrm{PR}(q)}{L(q)}$$

d is a damping factor (~0.85, modelling a "random surfer" who occasionally jumps to a random page instead of following links), N is the total node count, and the sum runs over every node q that links to p, weighted by PR(q) divided by q's total outgoing link count L(q) — so a link's "vote" is worth less if that node links to many other things.

In [ ]:
import numpy as np

# A tiny B2B referral network: Mehta Textiles' supplier/buyer graph
# nodes: 0=Mehta Textiles, 1=Cotton Supplier A, 2=Dye House B, 3=Export Buyer C, 4=Logistics D
# edges[i][j] = 1 means node i "recommends"/links to node j
edges = np.array([
    [0,1,1,0,1],
    [1,0,0,0,0],
    [1,0,0,1,0],
    [1,0,1,0,0],
    [1,0,0,0,0],
])
N = len(edges)
d = 0.85
pr = np.ones(N) / N   # start uniform
out_degree = edges.sum(axis=1)

for _ in range(100):   # power iteration — repeatedly propagate rank until convergence
    new_pr = np.ones(N) * (1-d)/N
    for p in range(N):
        for q in range(N):
            if edges[q][p] and out_degree[q] > 0:
                new_pr[p] += d * pr[q] / out_degree[q]
    pr = new_pr

names = ['Mehta Textiles','Cotton Supplier A','Dye House B','Export Buyer C','Logistics D']
for name, score in sorted(zip(names, pr), key=lambda x: -x[1]):
    print(f"  {name:20s}  PageRank={score:.4f}")

Mehta Textiles ranks highest not merely because it has the most outgoing links, but because it's the recurring target that other well-connected nodes point back to — the recursive "important nodes linking to you" effect. Beyond web search, this same algorithm is used for fraud-ring detection in transaction networks, influential-account detection in social graphs, and citation-importance ranking in academic literature.

## ⚠ Advanced: Fuzzy C-Means

K-Means forces every point into exactly one cluster (hard assignment) — the same limitation GMM was introduced to solve, but via a full probabilistic Gaussian model. **Fuzzy C-Means** offers a lighter-weight middle ground: each point gets a soft **membership degree** to every cluster, without requiring GMM's full covariance/Gaussian machinery.

$$\text{minimise } \sum_i\sum_j u_{ij}^m \lVert x_i - c_j \rVert^2 \qquad u_{ij} = \text{membership of point } i \text{ in cluster } j\ \left(\sum_j u_{ij}=1\right),\ m = \text{fuzziness exponent (typically 2)}$$

The fuzziness exponent m controls how "soft" the boundaries are: m → 1 approaches hard K-Means; larger m produces increasingly blended memberships. Unlike GMM, Fuzzy C-Means makes no distributional assumption about cluster shape — it's purely a fuzzified version of K-Means' distance-based objective, making it faster and simpler to reason about when GMM's Gaussian assumption feels like overkill.

In [ ]:
import numpy as np
from skfuzzy.cluster import cmeans

# Same Mumbai customer spending data used on the GMM page
data = X.T  # skfuzzy expects features-as-rows
centers, u, u0, d, jm, n_iter, fpc = cmeans(
    data, c=3, m=2, error=0.005, maxiter=1000)

# u is the (3, n_samples) membership matrix — soft, like GMM's responsibilities
print(f"Sample 0 fuzzy memberships: {np.round(u[:,0], 3)}")
print(f"Fuzzy Partition Coefficient (cluster distinctness, 1=hard, 1/c=totally fuzzy): {fpc:.3f}")

## ⚠ Advanced: Spectral Clustering

K-Means, GMM, and Fuzzy C-Means all rely fundamentally on distance-to-centroid — which fails badly on non-convex cluster shapes (two interleaved crescents, concentric rings). **Spectral Clustering** takes a completely different approach: build a similarity graph between points, then cluster based on the graph's structure rather than raw distances.

- Build a similarity graph (e.g., connect each point to its k nearest neighbours, weighted by closeness)

- Compute the graph Laplacian L = D − W (D = degree matrix, W = similarity/adjacency matrix)

- Find the k eigenvectors of L with the smallest eigenvalues — this embeds points into a new space where cluster structure becomes linearly separable

- Run ordinary K-Means in this new eigenvector space

In [ ]:
from sklearn.cluster import SpectralClustering, KMeans
from sklearn.datasets import make_moons

# Two interleaved crescent-shaped clusters — the textbook K-Means failure case
X_moons, y_true = make_moons(n_samples=300, noise=0.06, random_state=42)

km_labels = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X_moons)
sc_labels = SpectralClustering(n_clusters=2, affinity='nearest_neighbors',
                                n_neighbors=10, random_state=42).fit_predict(X_moons)

from sklearn.metrics import adjusted_rand_score
print(f"K-Means agreement with true labels:    {adjusted_rand_score(y_true, km_labels):.3f}")
print(f"Spectral Clustering agreement:          {adjusted_rand_score(y_true, sc_labels):.3f}")

K-Means essentially fails on this shape (its centroids land in the middle of both crescents, splitting each one in half rather than separating them) while Spectral Clustering recovers the true structure perfectly — the graph-based, local-connectivity view sees the crescents' shape directly, where centroid-distance methods only ever see two blobs of points regardless of their actual geometry.

## Try It — Run Power Iteration Yourself on the Mehta Textiles Network

This is the exact 5-node graph from the code above (arrows show who links to whom; a double-headed arrow means the link goes both ways). Step through power iteration one round at a time, or jump straight to convergence, and watch each node's circle grow with its live PageRank.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Density with a kernel estimate

Fit `KernelDensity(bandwidth=150)` on the bimodal `orders` and store in `d_mode` the density at 400 and in `d_dip` the density at 1300 (use `np.exp(kde.score_samples(...))`). The dip between the two modes should be much lower.

In [ ]:
import numpy as np
from sklearn.neighbors import KernelDensity
rng = np.random.default_rng(0)
orders = np.concatenate([rng.normal(400, 80, 300), rng.normal(2200, 300, 60)]).reshape(-1, 1)
d_mode = d_dip = None   # TODO


In [ ]:
try:
    check("mode is denser than the dip", d_mode > 5 * d_dip)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.neighbors import KernelDensity
rng = np.random.default_rng(0)
orders = np.concatenate([rng.normal(400, 80, 300), rng.normal(2200, 300, 60)]).reshape(-1, 1)
kde = KernelDensity(bandwidth=150).fit(orders)
d_mode, d_dip = np.exp(kde.score_samples([[400.0], [1300.0]]))

```

</details>

### Exercise 2 · Medium · Fuzzy memberships

Run fuzzy c-means (`skfuzzy.cluster.cmeans`, `c=2, m=2, error=0.005, maxiter=500, seed=0`) on the data (features as **rows**). Store the membership matrix in `u`, and set `sums_to_one` = each point's memberships sum to 1.

In [ ]:
import numpy as np
from skfuzzy.cluster import cmeans
rng = np.random.default_rng(1)
pts = np.vstack([rng.normal([0, 0], 0.5, (40, 2)), rng.normal([5, 5], 0.5, (40, 2))])
u = sums_to_one = None   # TODO


In [ ]:
try:
    check("membership matrix is 2 x 80", u.shape == (2, 80))
    check("memberships sum to 1 for every point", sums_to_one is True)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from skfuzzy.cluster import cmeans
rng = np.random.default_rng(1)
pts = np.vstack([rng.normal([0, 0], 0.5, (40, 2)), rng.normal([5, 5], 0.5, (40, 2))])
centers, u, *_ = cmeans(pts.T, c=2, m=2, error=0.005, maxiter=500, seed=0)
sums_to_one = bool(np.allclose(u.sum(axis=0), 1))

```

</details>

### Exercise 3 · Stretch · Spectral clustering unfolds crescents

K-means fails on two interleaved half-moons. Compare it with `SpectralClustering(n_clusters=2, affinity="nearest_neighbors", n_neighbors=10, random_state=0)` using the **adjusted Rand index** against the true labels. Store `ari_kmeans`, `ari_spectral`.

In [ ]:
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.datasets import make_moons
from sklearn.metrics import adjusted_rand_score
Xm, ym = make_moons(n_samples=300, noise=0.06, random_state=42)
ari_kmeans = ari_spectral = None   # TODO


In [ ]:
try:
    check("K-means is poor", ari_kmeans < 0.6)
    check("spectral recovers the moons", ari_spectral > 0.95)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.datasets import make_moons
from sklearn.metrics import adjusted_rand_score
Xm, ym = make_moons(n_samples=300, noise=0.06, random_state=42)
ari_kmeans = adjusted_rand_score(ym, KMeans(2, n_init=10, random_state=0).fit_predict(Xm))
ari_spectral = adjusted_rand_score(ym, SpectralClustering(n_clusters=2, affinity="nearest_neighbors", n_neighbors=10, random_state=0).fit_predict(Xm))

```

Spectral clustering clusters the *connectivity graph*, so shape-agnostic groups separate cleanly.

</details>

---
*Back to the course: **Machine Learning End To End → Advanced Clustering, Density & Graph Methods**.*